# Evaluation & Analysis
**Task 7 — Dariga Borasheva**

Loads all metric logs produced by the pipeline stages and produces:
1. Per-model IMDb (source) performance table
2. Zero-shot cross-domain (Tweet) performance table
3. Domain-gap summary (F1 drop)
4. Few-shot learning curve
5. Self-training adaptation comparison
6. Combined cross-model summary figure


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

LOGS    = Path('../logs')
FIGURES = Path('../figures')
FIGURES.mkdir(exist_ok=True, parents=True)

LABEL_NAMES = ['negative', 'neutral', 'positive']

PALETTE = {
    'LogisticRegression': '#4C72B0',
    'LinearSVC':          '#DD8452',
    'RoBERTa':            '#55A868',
    'Self-Trained':       '#C44E52',
}
STYLE = dict(dpi=150, bbox_inches='tight')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120


## 1. Load Logs


In [ ]:
def _load(path):
    if not path.exists():
        raise FileNotFoundError(f'Missing log: {path}')
    with open(path) as f:
        return json.load(f)

baseline   = _load(LOGS / 'training_and_testing_metrics.json')
finetune   = _load(LOGS / 'finetune_metrics.json')
few_shot   = _load(LOGS / 'few_shot_metrics.json')
self_train = _load(LOGS / 'self_training_metrics.json')

print('All logs loaded.')
print('  baseline   keys:', list(baseline.keys()))
print('  finetune   keys:', list(finetune.keys()))
print('  few_shot   runs:', len(few_shot))
print('  self_train keys:', list(self_train.keys()))


## 2. Source-Domain (IMDb) Performance


In [ ]:
rows = []
for model, splits in {**baseline['imdb'], **finetune['imdb']}.items():
    for split, m in splits.items():
        rows.append({'Model': model, 'Split': split,
                     'Accuracy': m['accuracy'], 'Macro-F1': m['macro_f1']})

imdb_df = (pd.DataFrame(rows)
             .set_index(['Model', 'Split'])
             .sort_index())
imdb_df


### IMDb Test Macro-F1 — Model Comparison


In [ ]:
test_df = imdb_df.xs('test', level='Split')
models  = test_df.index.tolist()
f1s     = test_df['Macro-F1'].tolist()
accs    = test_df['Accuracy'].tolist()

x, w = np.arange(len(models)), 0.35
fig, ax = plt.subplots(figsize=(8, 4))
colors = [PALETTE.get(m, '#888') for m in models]
ax.bar(x - w/2, f1s,  w, label='Macro-F1',  color=colors)
ax.bar(x + w/2, accs, w, label='Accuracy',   color=colors, alpha=0.5)
for i, (f, a) in enumerate(zip(f1s, accs)):
    ax.text(i-w/2, f+0.005, f'{f:.3f}', ha='center', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(models, rotation=15, ha='right')
ax.set_title('IMDb Test Performance by Model'); ax.set_ylim(0,1.05)
ax.set_ylabel('Score'); ax.legend(); ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / '1_imdb_test_performance.png', **STYLE)
plt.show()


## 3. Zero-Shot Tweet Performance


In [ ]:
best_all = {**baseline.get('best_thresholds',{}), **finetune.get('best_thresholds',{})}
all_zs   = {**baseline['tweet_zeroshot'],          **finetune['tweet_zeroshot']}
rows = []
for model, bt in best_all.items():
    zs = all_zs.get(model, {})
    key = bt if bt in zs else str(bt)
    if key in zs:
        rows.append({'Model': model, 'Best Threshold': bt,
                     'Accuracy': zs[key]['accuracy'],
                     'Macro-F1': zs[key]['macro_f1']})

zs_df = pd.DataFrame(rows).set_index('Model')
zs_df


In [ ]:
models_zs = zs_df.index.tolist()
f1s_zs    = zs_df['Macro-F1'].tolist()
fig, ax = plt.subplots(figsize=(7, 4))
colors_zs = [PALETTE.get(m, '#888') for m in models_zs]
bars = ax.bar(models_zs, f1s_zs, color=colors_zs, width=0.5)
for b in bars:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.008,
            f'{b.get_height():.3f}', ha='center', fontsize=10)
ax.set_title('Zero-Shot Tweet Macro-F1 (best threshold per model)')
ax.set_ylabel('Macro-F1'); ax.set_ylim(0,1.0); ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / '2_zeroshot_tweet_performance.png', **STYLE)
plt.show()


## 4. Domain Gap Summary


In [ ]:
all_imdb_test = {m: v['test']['macro_f1'] for m,v in {**baseline['imdb'],**finetune['imdb']}.items()}
rows_gap = []
for model, bt in best_all.items():
    zs = all_zs.get(model,{})
    key = bt if bt in zs else str(bt)
    if key not in zs or model not in all_imdb_test: continue
    src = all_imdb_test[model]; tgt = zs[key]['macro_f1']
    rows_gap.append({'Model':model,'IMDb F1':src,'Tweet F1 (zero-shot)':tgt,'Drop':round(src-tgt,4)})

gap_df = pd.DataFrame(rows_gap).set_index('Model')
gap_df.style.background_gradient(subset=['Drop'], cmap='RdYlGn_r')


In [ ]:
models_g = gap_df.index.tolist()
src_f1s  = gap_df['IMDb F1'].tolist()
tgt_f1s  = gap_df['Tweet F1 (zero-shot)'].tolist()
x, w = np.arange(len(models_g)), 0.35
fig, ax = plt.subplots(figsize=(8,4))
colors_g = [PALETTE.get(m,'#888') for m in models_g]
ax.bar(x-w/2, src_f1s, w, label='IMDb (source)',    color=colors_g, alpha=0.9)
ax.bar(x+w/2, tgt_f1s, w, label='Tweet (zero-shot)',color=colors_g, alpha=0.5)
for i,(s,t) in enumerate(zip(src_f1s,tgt_f1s)):
    ax.annotate(f'↓{s-t:.3f}', xy=(i, min(s,t)-0.04), ha='center',
                fontsize=9, color='crimson', fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(models_g, rotation=15, ha='right')
ax.set_title('Domain Gap: Source vs Target Macro-F1')
ax.set_ylabel('Macro-F1'); ax.set_ylim(0,1.05); ax.legend(); ax.grid(axis='y',alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / '3_domain_gap.png', **STYLE)
plt.show()


## 5. Few-Shot Learning Curve


In [ ]:
fs_df = pd.DataFrame(few_shot).sort_values('k')
fs_df['label'] = fs_df['k'].apply(lambda k: 'Zero-shot' if k==0 else f'K={k}')
fs_df.set_index('label')[['train_size','accuracy','macro_f1']]


In [ ]:
ks     = fs_df['k'].tolist()
f1s_fs = fs_df['macro_f1'].tolist()
accs_f = fs_df['accuracy'].tolist()
xlabels = ['Zero-shot' if k==0 else f'K={k}' for k in ks]
x_fs    = np.arange(len(ks))

fig, ax = plt.subplots(figsize=(9,4))
ax.plot(x_fs, f1s_fs, marker='o', label='Macro-F1', color='steelblue', linewidth=2)
ax.plot(x_fs, accs_f, marker='s', label='Accuracy',  color='darkorange', linewidth=2, linestyle='--')
for i,(k,f) in enumerate(zip(ks,f1s_fs)):
    ax.annotate(f'{f:.3f}', (i,f), textcoords='offset points', xytext=(0,8), ha='center', fontsize=9)
ax.set_xticks(x_fs); ax.set_xticklabels(xlabels)
ax.set_title('Few-Shot Learning Curve on Tweets (RoBERTa)')
ax.set_ylabel('Score'); ax.set_ylim(0,1.0); ax.legend(); ax.grid(axis='y',alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / '4_few_shot_learning_curve.png', **STYLE)
plt.show()


## 6. Self-Training Adaptation


In [ ]:
st_metrics = self_train['metrics']
st_df = pd.DataFrame(st_metrics).set_index('Model')
print(f"Accepted pseudo-labels: {self_train.get('accepted_samples','-'):,} / {self_train.get('total_unlabelled','-'):,}")
st_df


In [ ]:
st_models = st_df.index.tolist()
st_f1s    = st_df['Macro-F1'].tolist()
st_accs   = st_df['Accuracy'].tolist()
x, w = np.arange(len(st_models)), 0.35
colors_st = ['#4C72B0','#C44E52'][:len(st_models)]
fig, ax = plt.subplots(figsize=(7,4))
ax.bar(x-w/2, st_f1s, w, label='Macro-F1', color=colors_st)
ax.bar(x+w/2, st_accs,w, label='Accuracy',  color=colors_st, alpha=0.5)
for i,(f,a) in enumerate(zip(st_f1s,st_accs)):
    ax.text(i-w/2, f+0.01, f'{f:.3f}', ha='center', fontsize=9, fontweight='bold')
    ax.text(i+w/2, a+0.01, f'{a:.3f}', ha='center', fontsize=9)
if len(st_f1s)==2:
    delta=st_f1s[1]-st_f1s[0]; sign='+' if delta>=0 else ''
    ax.annotate(f'ΔF1 = {sign}{delta:.3f}', xy=(0.5, max(st_f1s)+0.06),
                xycoords=('data','data'), ha='center', fontsize=11,
                color='darkgreen' if delta>0 else 'crimson', fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(st_models)
ax.set_title('Self-Training: Before vs After')
ax.set_ylabel('Score'); ax.set_ylim(0,1.1); ax.legend(); ax.grid(axis='y',alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / '5_self_training_comparison.png', **STYLE)
plt.show()


## 7. Full Cross-Model Summary


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Sentiment Domain Shift: Full Evaluation Summary', fontsize=14, fontweight='bold')

# Panel 0 — IMDb test
ax = axes[0,0]
ax.bar(models, f1s, color=[PALETTE.get(m,'#888') for m in models], width=0.5)
for i,f in enumerate(f1s): ax.text(i, f+0.005, f'{f:.3f}', ha='center', fontsize=8)
ax.set_title('IMDb Test Macro-F1'); ax.set_ylim(0,1.05)
ax.tick_params(axis='x', rotation=15); ax.grid(axis='y',alpha=0.3)

# Panel 1 — Zero-shot tweet
ax = axes[0,1]
ax.bar(models_zs, f1s_zs, color=[PALETTE.get(m,'#888') for m in models_zs], width=0.5)
for i,f in enumerate(f1s_zs): ax.text(i, f+0.005, f'{f:.3f}', ha='center', fontsize=8)
ax.set_title('Zero-Shot Tweet Macro-F1'); ax.set_ylim(0,1.05)
ax.tick_params(axis='x', rotation=15); ax.grid(axis='y',alpha=0.3)

# Panel 2 — Few-shot curve
ax = axes[1,0]
ax.plot(x_fs, f1s_fs, marker='o', color='steelblue', linewidth=2)
for i,f in enumerate(f1s_fs):
    ax.annotate(f'{f:.3f}',(i,f),textcoords='offset points',xytext=(0,6),ha='center',fontsize=8)
ax.set_xticks(x_fs); ax.set_xticklabels(xlabels, fontsize=8)
ax.set_title('Few-Shot Learning Curve'); ax.set_ylim(0,1.0); ax.grid(axis='y',alpha=0.3)

# Panel 3 — Self-training
ax = axes[1,1]
bars3 = ax.bar(st_models, st_f1s, color=['#4C72B0','#C44E52'][:len(st_models)], width=0.4)
for b in bars3:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.005,
            f'{b.get_height():.3f}', ha='center', fontsize=9, fontweight='bold')
ax.set_title('Self-Training Before vs After'); ax.set_ylim(0,1.05)
ax.tick_params(axis='x', rotation=10); ax.grid(axis='y',alpha=0.3)

fig.tight_layout()
fig.savefig(FIGURES / '6_full_summary.png', **STYLE)
plt.show()


## 8. Save Evaluation Summary JSON


In [ ]:
summary = {
    'imdb_performance':     json.loads(imdb_df.to_json(orient='index')),
    'zeroshot_performance': json.loads(zs_df.to_json(orient='index')),
    'domain_gap':           json.loads(gap_df.to_json(orient='index')),
    'few_shot_curve':       few_shot,
    'self_training':        self_train['metrics'],
    'accepted_pseudo_labels': self_train.get('accepted_samples'),
    'total_unlabelled':       self_train.get('total_unlabelled'),
}
out = LOGS / 'evaluation_summary.json'
with open(out, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Saved → {out}')
